In [1]:
# --- Guarda de entorno --------------------------------------------------------
# Falla rápido y con un mensaje claro en vez de dejar que PySpark reviente 20
# líneas más abajo con un error que no menciona la causa real.
import os, shutil, sys

problemas = []

if not os.path.isdir("/opt/spark"):
    problemas.append("no existe /opt/spark: no estás en el contenedor del clúster")

if shutil.which("java") is None:
    problemas.append("no hay Java en el PATH: PySpark no puede arrancar la JVM")

try:
    import pyspark
    if not pyspark.__file__.startswith("/opt/spark/"):
        problemas.append(
            f"pyspark viene de {pyspark.__file__}\n"
            f"        (instalado por pip/conda) en vez de /opt/spark/python. "
            f"Esa copia no está alineada con el clúster.")
except ImportError:
    problemas.append("pyspark no es importable")

if problemas:
    print("=" * 72)
    print("ESTE NOTEBOOK NO ESTÁ CORRIENDO EN EL CLÚSTER")
    print("=" * 72)
    for p in problemas:
        print(f"  - {p}")
    print()
    print("  Ábrelo en el Jupyter DEL CLÚSTER:")
    print("      http://localhost:8889        (token: lab-sitio3)")
    print()
    print("  El del puerto 8888 es el entorno de la Fase 1. Monta esta misma")
    print("  carpeta, por eso ves el mismo archivo, pero no tiene Java ni")
    print("  conexión con los workers.")
    print("=" * 72)
    raise SystemExit("Entorno incorrecto: usa http://localhost:8889")

print(f"Entorno correcto")
print(f"  Python  : {sys.version.split()[0]}")
print(f"  Java    : {shutil.which('java')}")
print(f"  PySpark : {pyspark.__file__}")


Entorno correcto
  Python  : 3.10.12
  Java    : /opt/java/openjdk/bin/java
  PySpark : /opt/spark/python/pyspark/__init__.py


In [10]:
import os, time
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, LongType, TimestampType)

spark = (
    SparkSession.builder
    .appName("fase2-sitio3")
    .master(os.environ.get("SPARK_MASTER_URL", "spark://spark-master:7077"))
    # --- los tres parámetros críticos en Docker ---
    .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", "jupyter"))
    .config("spark.driver.bindAddress", "0.0.0.0")
    # --- memoria: debe caber en lo que declara cada worker (ver .env) ---
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "2g"))
    .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "2g"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
print(f"Spark          : {spark.version}")
print(f"Master         : {sc.master}")
print(f"Executors      : {sc._jsc.sc().getExecutorMemoryStatus().size() - 1}")
print(f"Paralelismo    : {sc.defaultParallelism} tareas simultáneas")
print(f"UI del driver  : http://localhost:4040")



Spark          : 4.0.1
Master         : spark://spark-master:7077
Executors      : 2
Paralelismo    : 4 tareas simultáneas
UI del driver  : http://localhost:4040


In [1]:
from pyspark.sql import SparkSession, functions as F 
 
spark = ( 
    SparkSession.builder 
    .appName("fase2-sitio3") 
    .master("spark://spark-master:7077") 
    .config("spark.driver.host", "jupyter") 
    .config("spark.driver.bindAddress", "0.0.0.0") 
    .getOrCreate() 
) 

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 21:55:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.json("/opt/workspace/data/raw/*.jsonl")   # no lee nada todavía 
limpio = df.filter(F.col("value") > -18)                  # no filtra nada todavía 
limpio.count()                                            # AHORA se ejecuta todo

26/10/06 21:55:45 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /opt/workspace/data/raw/*.jsonl.
java.io.FileNotFoundException: File /opt/workspace/data/raw/*.jsonl does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalys

0

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType 
 
esquema = StructType([ 
    StructField("timestamp", StringType(), True), 
    StructField("sensor_id", StringType(), True), 
    StructField("metric",    StringType(), True), 
    StructField("value",     DoubleType(), True), 
    StructField("unit",      StringType(), True), 
    StructField("worker_id", LongType(),   True), 
]) 
 
df = spark.read.schema(esquema).json("/opt/workspace/data/raw/*.jsonl") 

In [3]:
import os, json, random
from datetime import datetime, timedelta

raw_dir = "/opt/workspace/data/raw"
os.makedirs(raw_dir, exist_ok=True)

# Limpiar la muestra unitaria previa si existe
if os.path.exists(f"{raw_dir}/muestra.jsonl"):
    os.remove(f"{raw_dir}/muestra.jsonl")

sensores = [f"sensor_{i}" for i in range(1, 5)]  # 4 sensores base
metricas = [
    ("temperature", "C", -25.0, 5.0),
    ("pressure", "bar", 1.0, 10.0),
    ("vibration", "mm/s", 0.1, 4.5)
]

num_archivos = 5
filas_por_archivo = 50000  # 250.000 filas en total (rápido de generar y suficiente para Spark)
base_time = datetime(2026, 10, 1, 0, 0, 0)

print("Generando archivos de telemetría en data/raw...")

for arch_idx in range(num_archivos):
    file_path = f"{raw_dir}/telemetria_part_{arch_idx}.jsonl"
    with open(file_path, "w", encoding="utf-8") as f:
        for i in range(filas_por_archivo):
            metric, unit, v_min, v_max = random.choice(metricas)
            reg = {
                "timestamp": (base_time + timedelta(seconds=(arch_idx * filas_por_archivo + i))).isoformat() + "Z",
                "sensor_id": random.choice(sensores),
                "metric": metric,
                "value": round(random.uniform(v_min, v_max), 3),
                "unit": unit,
                "worker_id": random.choice([1, 2])
            }
            f.write(json.dumps(reg) + "\n")
    print(f" -> Creado: telemetria_part_{arch_idx}.jsonl ({filas_por_archivo:,} filas)")

print("\n¡Listo! Archivos en data/raw:", os.listdir(raw_dir))

Generando archivos de telemetría en data/raw...
 -> Creado: telemetria_part_0.jsonl (50,000 filas)
 -> Creado: telemetria_part_1.jsonl (50,000 filas)
 -> Creado: telemetria_part_2.jsonl (50,000 filas)
 -> Creado: telemetria_part_3.jsonl (50,000 filas)
 -> Creado: telemetria_part_4.jsonl (50,000 filas)

¡Listo! Archivos en data/raw: ['telemetria_part_0.jsonl', 'telemetria_part_1.jsonl', 'telemetria_part_2.jsonl', 'telemetria_part_3.jsonl', 'telemetria_part_4.jsonl']


In [4]:
df = spark.read.json("/opt/workspace/data/raw")
print(f"Total registros: {df.count():,}")
print(f"Particiones: {df.rdd.getNumPartitions()}")
df.show(5)

Total registros: 250,000
Particiones: 5
+-----------+---------+--------------------+----+-------+---------+
|     metric|sensor_id|           timestamp|unit|  value|worker_id|
+-----------+---------+--------------------+----+-------+---------+
|temperature| sensor_1|2026-10-03T07:33:20Z|   C|-16.851|        1|
|   pressure| sensor_3|2026-10-03T07:33:21Z| bar|  5.308|        2|
|temperature| sensor_1|2026-10-03T07:33:22Z|   C|-23.673|        2|
|temperature| sensor_1|2026-10-03T07:33:23Z|   C|  2.421|        2|
|   pressure| sensor_1|2026-10-03T07:33:24Z| bar|  3.947|        2|
+-----------+---------+--------------------+----+-------+---------+
only showing top 5 rows


In [5]:
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# 1. Crear sesión asegurando visibilidad bidireccional entre Jupyter y los Workers
spark = (
    SparkSession.builder
    .appName("fase2-sitio3")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

print(f"Sesión Spark {spark.version} lista y conectada.")

Sesión Spark 4.0.1 lista y conectada.


26/10/06 22:13:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
RUTA_CRUDA = "/opt/workspace/data/raw"

# --- Opción A: inferSchema ---
ini = time.time()
df_inferido = spark.read.json(RUTA_CRUDA)
esquema_inferido = df_inferido.schema
t_inferencia = time.time() - ini

print(f"Inferir el esquema tardó: {t_inferencia:.2f} s")
df_inferido.printSchema()

Inferir el esquema tardó: 0.87 s
root
 |-- metric: string (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: double (nullable = true)
 |-- worker_id: long (nullable = true)

